# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Both findings below are from `docs/flyrank-seo-research-march-2026.pdf`, ML Appendix (pages 27 and 29). Questions are meant constructively — the paper is explicit that its ML pages are "exploratory appendix material," not headline evidence, so these are the kind of follow-up questions that section is inviting, not a takedown.

**Finding — "What Predicts Health?" (p.27):** a Random Forest ranks feature importance for predicting `health_score`, finding Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top drivers.

- *Where does the label come from?* `health_score` is explicitly defined elsewhere in the paper as `impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)` — a formula built directly from three of the same signals the model is "discovering" as important. The paper itself flags this ("the target itself is partly constructed from some of these inputs"), which is the right instinct — but the reported importances (position 43%, impressions 32%, scroll 15%) line up closely with the scoring formula's own weights (30/30/20/20 pts). **My question:** how much of this "prediction" is really just the model re-deriving its own scoring formula, versus finding something the formula didn't already encode? A cleaner test would be predicting a genuinely external outcome (e.g. future traffic change) rather than a score partly built from the same inputs.
- *Does the validation design carry the claim?* The methodology page states an 80/20 split for the Random Forest, but doesn't say whether it's grouped by brand. With 57 brands in the dataset, a plain random row split risks letting pages from the same brand appear in both train and test — which is exactly the risk I test for my own model in Section 2 below.

**Finding — "What Predicts Growth?" (p.29):** a Logistic Regression reports 71% holdout accuracy separating growing from declining pages, with Content Age as the strongest negative coefficient.

- *Where does the label come from?* The paper doesn't state the exact growing/declining definition in the text I could read, but it's very likely a same-window trend calculation (comparable to the starter pipeline's own `trend_direction` field) — a proxy computed from the current data, not a validated future outcome. **My question:** is "growing/declining" defined from a trailing comparison (e.g. last-30 vs. prior-30, like the portfolio trend cards elsewhere in the paper) that could overlap with when the features were measured? If so, 71% accuracy describes how well the model recovers its own label-defining calculation, not necessarily a forward-looking growth prediction.
- *Does the validation design carry the claim?* Same open question as above — an 80/20 split is stated, but nothing confirms it's grouped by brand, which matters even more here since "growth" patterns likely cluster by brand/vertical.

In [ ]:
# No query needed here -- Section 1 is a close reading of the paper itself, with
# specific page references. Section 2 below tests the exact same methodology question
# (grouped vs ungrouped split) on my own model, rather than just speculating about the paper's.

## 2. My model under an honest split (before/after)

Re-running my Week 5 model under a **naive, ungrouped 80/20 split** ("before") against the **client-grouped split I actually used** ("after") — the same methodology gap I just flagged in the paper's own ML appendix, tested on my own work instead of just speculating about someone else's.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 160)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()

lane4['expected_ctr_for_tier'] = lane4.groupby('position_tier')['ctr'].transform('median')
lane4['ctr_gap_score'] = (lane4['expected_ctr_for_tier'] - lane4['ctr']).clip(lower=0)
median_scroll = lane4['scroll_rate'].median()
lane4['engagement_deficit'] = ((lane4['engagement_rate'] == 0) & (lane4['scroll_rate'] < median_scroll)).astype(int)
lane4['log_impressions'] = np.log1p(lane4['impressions_90d'])
lane4['word_count_missing'] = lane4['word_count'].isna().astype(int)
lane4['word_count_filled'] = lane4['word_count'].fillna(-1)

cat_cols = ['position_tier', 'competition_level', 'main_intent']
num_cols = ['avg_position', 'log_impressions', 'ctr', 'content_age_days', 'word_count_filled', 'word_count_missing']
dummies = pd.get_dummies(lane4[cat_cols], columns=cat_cols)
lane4_enc = pd.concat([lane4[num_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
extra = lane4[['client_id', 'engagement_deficit', 'ctr_gap_score']].reset_index(drop=True)
lane4_enc = pd.concat([lane4_enc, extra], axis=1)
feature_cols = [c for c in lane4_enc.columns if c not in ['client_id', 'engagement_deficit', 'ctr_gap_score']]

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random row-level split -- ignores client grouping entirely
Xtr, Xte, ytr, yte, cl_tr, cl_te = train_test_split(
    lane4_enc[feature_cols], lane4_enc['engagement_deficit'], lane4_enc['client_id'],
    test_size=0.25, random_state=42, stratify=lane4_enc['engagement_deficit'])
overlap_naive = set(cl_tr) & set(cl_te)

m_naive = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                  n_jobs=-1, class_weight='balanced').fit(Xtr, ytr)
probs_naive = m_naive.predict_proba(Xte)[:, 1]

print('=== BEFORE: naive random row-level split ===')
print(f'client overlap: {len(overlap_naive)} of {lane4["client_id"].nunique()} total clients appear in BOTH train and test')
print(f'ROC AUC: {roc_auc_score(yte, probs_naive):.3f}   '
      f'precision@20: {precision_at_k(probs_naive, yte.values, 20):.3f}   '
      f'precision@50: {precision_at_k(probs_naive, yte.values, 50):.3f}')
print()

# AFTER: honest client-grouped split -- the same one used in Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(lane4_enc, groups=lane4_enc['client_id']))
train, test = lane4_enc.iloc[train_idx], lane4_enc.iloc[test_idx]

m_grouped = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                    n_jobs=-1, class_weight='balanced').fit(train[feature_cols], train['engagement_deficit'])
probs_grouped = m_grouped.predict_proba(test[feature_cols])[:, 1]

print('=== AFTER: honest client-grouped split (Week 5\'s actual split) ===')
overlap_grouped = set(lane4_enc['client_id'].iloc[train_idx]) & set(lane4_enc['client_id'].iloc[test_idx])
print(f'client overlap: {len(overlap_grouped)} clients')
print(f'ROC AUC: {roc_auc_score(test["engagement_deficit"], probs_grouped):.3f}   '
      f'precision@20: {precision_at_k(probs_grouped, test["engagement_deficit"].values, 20):.3f}   '
      f'precision@50: {precision_at_k(probs_grouped, test["engagement_deficit"].values, 50):.3f}')

=== BEFORE: naive random row-level split ===
client overlap: 27 of 28 total clients appear in BOTH train and test
ROC AUC: 0.743   precision@20: 0.750   precision@50: 0.680

=== AFTER: honest client-grouped split (Week 5's actual split) ===
client overlap: 0 clients
ROC AUC: 0.776   precision@20: 0.600   precision@50: 0.520


**Reading this honestly:** with 27 of 28 clients appearing in both train and test, the naive split's precision@20 (0.750) and precision@50 (0.680) are both noticeably higher than the grouped split's real numbers (0.600 and 0.520) — the inflation I expected, and the exact same risk I flagged in the paper's own methodology section above. One genuine surprise worth reporting rather than smoothing over: ROC AUC did **not** show the same clean inflation (0.743 naive vs. 0.776 grouped — actually slightly *lower* under the leaky split, on this particular run). That's a useful reminder that leakage doesn't inflate every metric predictably — precision@K, which is what this project's actual decision depends on, showed the expected distortion clearly; a different metric might not, and checking only one metric could have missed this.

In [2]:
# No additional query needed here -- the reading above is grounded entirely in the
# printed before/after numbers from the previous cell.

## 3. Leakage audit

The same hunt from Week 3, repeated on the final feature set actually used in Week 5's model.

In [3]:
# Attack 1: does any single final feature correlate suspiciously with the label
# (near 1.0, the way ctr_mar/LEAKY_ctr_bucket did against is_low_ctr in Week 3)?
final_features = ['avg_position', 'log_impressions', 'ctr', 'content_age_days',
                   'word_count_filled', 'word_count_missing']
corrs = lane4[final_features + ['engagement_deficit']].corr()['engagement_deficit'].drop('engagement_deficit')
corrs = corrs.reindex(corrs.abs().sort_values(ascending=False).index)
print('=== ATTACK 1: correlation of each final feature with the label ===')
print(corrs.round(3))
print(f'max abs correlation: {corrs.abs().max():.3f} -- well below the near-1.0 that flagged the ')
print('deliberately-leaky column in Week 3. No single feature is silently restating the label.')
print()

# Attack 2: engagement_rate / scroll_rate must NOT be in the final feature list at all
# (they define the label -- including them would be the same mistake as ctr_mar/is_low_ctr)
banned_from_features = {'engagement_rate', 'scroll_rate'}
print('=== ATTACK 2: banned label-defining columns in the final feature set ===')
print('overlap:', banned_from_features & set(final_features), '-- must be empty')
print()

# Attack 3: product-decision flags anywhere in the source data
banned_flags = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
hits = [c for c in df.columns if any(b in c.lower() for b in banned_flags)]
print('=== ATTACK 3: product-decision-flag columns in the starter CSV ===')
print(hits if hits else 'none found -- nothing to have leaked in.')

=== ATTACK 1: correlation of each final feature with the label ===
log_impressions      -0.292
ctr                  -0.217
word_count_missing    0.168
word_count_filled    -0.147
content_age_days      0.069
avg_position          0.044
Name: engagement_deficit, dtype: float64
max abs correlation: 0.292 -- well below the near-1.0 that flagged the 
deliberately-leaky column in Week 3. No single feature is silently restating the label.

=== ATTACK 2: banned label-defining columns in the final feature set ===
overlap: set() -- must be empty

=== ATTACK 3: product-decision-flag columns in the starter CSV ===
none found -- nothing to have leaked in.


## 4. Claim rewrite

**My boldest sentence, from Week 5:** *"The model clearly beats the baseline — precision@20 of 0.600 vs. the baseline's 0.350, and ROC AUC of 0.776 vs. 0.648."*

**Rewritten in safe language:** *"On a held-out set of clients the model never saw during training, a tier-adjusted random forest ranked engagement-troubled pages more precisely than the flat baseline rule — roughly 60% of its top 20 picks showed the independent engagement-deficit signal, versus 35% for the baseline, on this one 30,000-row anonymized sample. This is a directional, decision-support result on a single validated split, not a guarantee it holds at warehouse scale, and roughly 40% of the model's advantage traces to a single feature (impression volume) the baseline never had access to — so part of the gap is a fairer comparison problem, not purely a smarter model."*

The rewrite doesn't just add hedges — it removes the unqualified "clearly beats," names the actual sample and validation conditions, and states plainly that some of the gain has a mundane explanation (the baseline lacked a feature) rather than letting "the model is smarter" stand unchallenged.

In [4]:
# No query needed here -- this section is a rewriting exercise grounded in numbers
# already established and verified in Sections 2 and 3, and in Week 5's own notebook.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.